# 01_preprocess_GSE40279

This notebook preprocesses the GEO dataset **GSE40279** for Horvath DNAmAge
replication.

Objectives:
- inspect the raw GEO series matrix
- construct a CpG × Sample DNA methylation β-value matrix
- extract sample metadata (age, gender)
- export processed data for downstream analysis and pipeline execution

## 1. Setup

Define project paths and load required libraries.
Paths are defined once here and reused throughout the notebook.

In [2]:
import gzip
from pathlib import Path
import pandas as pd
import GEOparse

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw" / "GSE40279"
PROC_DIR = ROOT / "data" / "processed"

path = RAW_DIR / "GSE40279_series_matrix.txt.gz"
gse = GEOparse.get_GEO(geo="GSE40279", destdir=RAW_DIR)

04-Jan-2026 18:37:10 DEBUG utils - Directory /Volumes/Extreme_Pro/mac/Coding/m_clock/methylation-clock-replication/data/raw/GSE40279 already exists. Skipping.
04-Jan-2026 18:37:10 INFO GEOparse - File already exist: using local version.
04-Jan-2026 18:37:10 INFO GEOparse - Parsing /Volumes/Extreme_Pro/mac/Coding/m_clock/methylation-clock-replication/data/raw/GSE40279/GSE40279_family.soft.gz: 
04-Jan-2026 18:37:10 DEBUG GEOparse - DATABASE: GeoMiame
04-Jan-2026 18:37:10 DEBUG GEOparse - SERIES: GSE40279
04-Jan-2026 18:37:10 DEBUG GEOparse - PLATFORM: GPL13534
/Volumes/Extreme_Pro/mac/Coding/conda_envs/methylation-clock/lib/python3.11/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (11,14,15,36) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
04-Jan-2026 18:37:13 DEBUG GEOparse - SAMPLE: GSM989827
04-Jan-2026 18:37:13 DEBUG GEOparse - SAMPLE: GSM989828
04-Jan-2026 18:37:13 DEBUG GEOparse -

## 2. Load GEO series matrix table

The GEO series matrix file contains metadata and the methylation table
within the same text file.

This section parses the file by locating the
`!series_matrix_table_begin` and `!series_matrix_table_end` markers,
and extracts only the table portion into a DataFrame.

In [3]:
header = None
data_lines = []

with gzip.open(path, "rt") as f:
    for line in f:
        line = line.rstrip("\n")
        if line.startswith("!series_matrix_table_begin"):
            header_line = f.readline().rstrip("\n")
            header = header_line.split("\t")
            break

    if header is None:
        raise ValueError("!series_matrix_table_begin が見つかりませんでした。series matrix ではない可能性があります。")

    for line in f:
        line = line.rstrip("\n")
        if line.startswith("!series_matrix_table_end"):
            break
        data_lines.append(line.split("\t"))

df_raw = pd.DataFrame(data_lines, columns=header)
df_raw.head(), df_raw.shape

(       "ID_REF" "GSM989827" "GSM989828" "GSM989829" "GSM989830" "GSM989831"  \
 0  "cg00000029"   0.4641974   0.4548833   0.4857639   0.4807854   0.5012195   
 1  "cg00000108"   0.9410907   0.9390332    0.918802   0.9299082   0.9345481   
 2  "cg00000109"   0.9111821   0.5964548   0.8703333   0.8896887   0.8904501   
 3  "cg00000165"   0.1320137   0.2069167   0.1628613   0.1977801   0.1484374   
 4  "cg00000236"   0.7178611   0.7239354   0.7191964    0.704061   0.7549129   
 
   "GSM989832" "GSM989833" "GSM989834" "GSM989835"  ... "GSM990618"  \
 0   0.4999175   0.4858518   0.5124422   0.5181552  ...   0.5609581   
 1   0.9505427    0.925855   0.9413304   0.9385279  ...   0.9346989   
 2   0.8984932   0.8939723   0.8920096   0.9008406  ...   0.8819568   
 3   0.2240929   0.4004885   0.1945532   0.1347103  ...   0.1998829   
 4   0.8291917   0.7237817   0.6951424    0.731872  ...   0.7590112   
 
   "GSM990619" "GSM990620" "GSM990621" "GSM990622" "GSM990623" "GSM990624"  \
 0   0.47208

In [4]:
df_raw.head()

,"""ID_REF""","""GSM989827""","""GSM989828""","""GSM989829""","""GSM989830""","""GSM989831""","""GSM989832""","""GSM989833""","""GSM989834""","""GSM989835""",...,"""GSM990618""","""GSM990619""","""GSM990620""","""GSM990621""","""GSM990622""","""GSM990623""","""GSM990624""","""GSM990625""","""GSM990626""","""GSM990627"""
0,"""cg00000029""",0.4641974,0.4548833,0.4857639,0.4807854,0.5012195,0.4999175,0.4858518,0.5124422,0.5181552,...,0.5609581,0.4720809,0.5085015,0.5051925,0.4434114,0.5274959,0.5883314,0.3629945,0.499145,0.4586001
1,"""cg00000108""",0.9410907,0.9390332,0.918802,0.9299082,0.9345481,0.9505427,0.925855,0.9413304,0.9385279,...,0.9346989,0.9786119,0.9220243,0.9630517,0.9926312,0.9581732,0.9824496,0.9543923,0.9316905,0.9747313
2,"""cg00000109""",0.9111821,0.5964548,0.8703333,0.8896887,0.8904501,0.8984932,0.8939723,0.8920096,0.9008406,...,0.8819568,0.9262886,0.930091,0.9465466,0.929131,0.922034,0.855145,0.9271835,0.9009384,0.8298687
3,"""cg00000165""",0.1320137,0.2069167,0.1628613,0.1977801,0.1484374,0.2240929,0.4004885,0.1945532,0.1347103,...,0.1998829,0.1651165,0.2102482,0.1773514,0.1187421,0.2230683,0.1621798,0.1964298,0.1674773,0.170578
4,"""cg00000236""",0.7178611,0.7239354,0.7191964,0.704061,0.7549129,0.8291917,0.7237817,0.6951424,0.731872,...,0.7590112,0.7928829,0.7303665,0.7838298,0.7870888,0.7789587,0.7968685,0.7130197,0.7302154,0.7828441


## 3. Construct β-value matrix (CpG × Sample)

The parsed table contains:
- one column representing CpG probe IDs (`ID_REF`)
- multiple columns corresponding to individual samples

In this step, CpG IDs are set as the index to construct a
CpG × Sample β-value matrix.

In [5]:
# Remove quotes from column names
df_raw.columns = df_raw.columns.str.replace('"', '', regex=False)

# Rename CpG column
df_raw = df_raw.rename(columns={"ID_REF": "CpG"})

# Remove quotes from CpG IDs
df_raw["CpG"] = df_raw["CpG"].str.replace('"', '', regex=False)

# Create CpG × Sample beta matrix
df_beta = df_raw.set_index("CpG")

df_beta.iloc[:5, :5], df_beta.shape

(            GSM989827  GSM989828  GSM989829  GSM989830  GSM989831
 CpG                                                              
 cg00000029  0.4641974  0.4548833  0.4857639  0.4807854  0.5012195
 cg00000108  0.9410907  0.9390332   0.918802  0.9299082  0.9345481
 cg00000109  0.9111821  0.5964548  0.8703333  0.8896887  0.8904501
 cg00000165  0.1320137  0.2069167  0.1628613  0.1977801  0.1484374
 cg00000236  0.7178611  0.7239354  0.7191964   0.704061  0.7549129,
 (473034, 656))

## 4. Build sample metadata (age and gender)

Sample-level metadata are extracted from GEO GSM records.

This section:
- parses the `characteristics_ch1` field
- extracts age and gender information
- constructs a sample metadata table indexed by sample ID

In [6]:
metadata_rows = []

def extract_age_gender(gsm):
    chars = gsm.metadata.get("characteristics_ch1", [])

    age = None
    gender = None

    for c in chars:
        c_low = c.lower()
        if c_low.startswith("age"):
            age = c.split(":")[1].strip()
        elif c_low.startswith("gender"):
            gender = c.split(":")[1].strip()

    return age, gender

rows = []

for gsm_id, gsm in gse.gsms.items():
    age, gender = extract_age_gender(gsm)

    rows.append({
        "sample_id": gsm_id,
        "age": age,
        "gender": gender
    })

sample_df = pd.DataFrame(rows).set_index("sample_id")
sample_df.head()

,age,gender
sample_id,,
GSM989827,67,F
GSM989828,89,F
GSM989829,66,F
GSM989830,64,F
GSM989831,62,F


## 5. Save metadata and quick checks

Convert age values to numeric format, inspect the age distribution,
and save the sample metadata table to the processed data directory.

In [7]:
out_path = PROC_DIR / "GSE40279_sample_metadata.csv"
sample_df.to_csv(out_path)

sample_df["age"] = pd.to_numeric(sample_df["age"], errors="coerce")
sample_df["age"].describe()

count    656.000000
mean      64.035061
std       14.736681
min       19.000000
25%       54.000000
50%       65.000000
75%       75.000000
max      101.000000
Name: age, dtype: float64

## 6. Consistency check between β matrix and metadata

Verify that sample IDs in the β-value matrix and the sample metadata
table are consistent and overlapping as expected.

In [8]:
samples_matrix = set(df_raw.columns)
samples_meta = set(sample_df.index)

print("in matrix:", len(samples_matrix))
print("in metadata:", len(samples_meta))
print("overlap:", len(samples_matrix & samples_meta))

in matrix: 657
in metadata: 656
overlap: 656


## 7. Export processed β-value matrix

Save the processed CpG × Sample β-value matrix for use in
downstream Horvath DNAmAge calculation and automated pipelines.

In [9]:
beta_out = PROC_DIR / "GSE40279_beta_for_R.csv"
df_beta.to_csv(beta_out)
beta_out

PosixPath('/Volumes/Extreme_Pro/mac/Coding/m_clock/methylation-clock-replication/data/processed/GSE40279_beta_for_R.csv')

## Notes

This notebook focuses on **data understanding and preprocessing**.

Model implementation and age prediction are handled in subsequent
notebooks and in the automated pipeline under `src/mclock/`.

---

## Appendix: Exploratory inspection of the raw GEO series matrix

This appendix documents exploratory inspection of the raw GEO
`series_matrix` file.

The purpose of this section is to:
- visually inspect the file structure
- confirm metadata placement
- understand how the methylation table is embedded in the file

The code in this section is **not part of the main preprocessing pipeline**.
It is preserved solely to provide transparency and justification for
the parsing strategy used above.

In [10]:
with gzip.open(path, "rt") as f:
    for _ in range(20):
        print(f.readline().rstrip())

!Series_title	"Genome-wide Methylation Profiles Reveal Quantitative Views of Human Aging Rates"
!Series_geo_accession	"GSE40279"
!Series_status	"Public on Nov 21 2012"
!Series_submission_date	"Aug 21 2012"
!Series_last_update_date	"Jul 06 2022"
!Series_pubmed_id	"23177740"
!Series_summary	"Genome wide DNA methylation profiling of individuals across a large age range. The Illumina Infinium 450k Human DNA methylation Beadchip was used to obtain DNA methylation profiles across approximately 450k CpGs from human whole blood."
!Series_overall_design	"Bisulphite converted DNA from the 656 samples were hybridised to the Illumina Infinium 450k Human Methylation Beadchip"
!Series_type	"Methylation profiling by array"
!Series_contributor	"K,,Zhang"
!Series_contributor	"T,,Ideker"
!Series_sample_id	"GSM989827 GSM989828 GSM989829 GSM989830 GSM989831 GSM989832 GSM989833 GSM989834 GSM989835 GSM989836 GSM989837 GSM989838 GSM989839 GSM989840 GSM989841 GSM989842 GSM989843 GSM989844 GSM989845 GSM989846 